# 03 · Silver Layer Test Suite

End-to-end quality gate for all 5 Silver tables produced by `08_silver_transformation`.

| Suite | What it checks |
|---|---|
| **T1 Reconciliation** | Every Bronze row arrives in Silver or Quarantine (row counts balance) |
| **T2 Row-to-Row Integrity** | Every Silver row is byte-for-byte traceable to a Bronze row (SHA-256) |
| **T3 Audit Columns** | All audit metadata columns present and non-null in Silver |

> **Zero-touch registry:** add a new Silver table by appending one dict to `REGISTRY`. No test code changes needed.


In [0]:
# ── 1. CONFIGURATION ─────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("project_catalog", "vstone_catalog", "1. Catalog")
dbutils.widgets.text("bronze_schema",   "bronze",         "2. Bronze Schema")
dbutils.widgets.text("silver_schema",   "silver",         "3. Silver Schema")

CATALOG = dbutils.widgets.get("project_catalog")
BRONZE  = dbutils.widgets.get("bronze_schema")
SILVER  = dbutils.widgets.get("silver_schema")

B = lambda t: f"{CATALOG}.{BRONZE}.{t}"
S = lambda t: f"{CATALOG}.{SILVER}.{t}"

print(f"Catalog : {CATALOG}")
print(f"Bronze  : {BRONZE}")
print(f"Silver  : {SILVER}")


In [0]:
# ── 2. TRANSFORM HELPERS ──────────────────────────────────────────────────────

def _norm(expr):
    """Row-level normaliser applied identically to Bronze and Silver before SHA-256.
    Collapses whitespace, lowercases, and replaces null with a sentinel string so
    both sides hash identically regardless of null representation."""
    return F.coalesce(
        F.lower(F.regexp_replace(F.trim(expr.cast("string")), r"\s+", " ")),
        F.lit("null_placeholder")
    )

# ── ID casts ──────────────────────────────────────────────────────────────────
def _id_main(col): return F.expr(f"try_cast(`{col}` as long)").cast("string")   # listings_silver_merged
def _id_dbl(col):  return F.col(f"`{col}`").cast("double").cast("long").cast("string")  # text / photo

# ── String UDFs (must match pandas behaviour in pipeline) ─────────────────────
def _std(col):   # standardize_text: lower + strip; null → "none"
    return F.when(F.col(f"`{col}`").isNull(), F.lit("none"))              .otherwise(F.lower(F.trim(F.col(f"`{col}`"))))

def _clean(col): # clean_text: strip only; null → "None"
    return F.when(F.col(f"`{col}`").isNull(), F.lit("None"))              .otherwise(F.trim(F.col(f"`{col}`")))

def _geo(col):   # standardize_geo: strip only; null → "None"  (same as _clean)
    return F.when(F.col(f"`{col}`").isNull(), F.lit("None"))              .otherwise(F.trim(F.col(f"`{col}`")))

# ── Passthrough / simple casts ────────────────────────────────────────────────
def _pass(col):  return F.col(f"`{col}`")
def _dbl(col):   return F.col(f"`{col}`").cast("double")
def _coal(col):  return F.coalesce(F.col(f"`{col}`").cast("string"), F.lit(""))

# ── Numeric / domain transforms ───────────────────────────────────────────────
def _price(col):   return F.expr(f"try_cast(regexp_replace(`{col}`, '[^0-9.]', '') as double)")
def _int2(col):    return F.expr(f"try_cast(try_cast(`{col}` as double) as int)")
def _date(col):
    return F.coalesce(
        F.try_to_timestamp(F.col(f"`{col}`"), F.lit("dd.MM.yyyy")),
        F.try_to_timestamp(F.col(f"`{col}`"), F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'"))
    )
def _eng_vol(col): return F.expr(f"try_cast(regexp_replace(regexp_replace(`{col}`,' л',''),',','.') as double)")
def _eng_pow(col): return F.expr(f"try_cast(regexp_replace(`{col}`,' л.с.','') as int)")

print("Transform helpers loaded ✓")


In [0]:
# ── 3. TABLE REGISTRY ─────────────────────────────────────────────────────────
# One dict per Silver table. Add new tables here — test functions are generic.

REGISTRY = [
    # ── listings_silver_merged ────────────────────────────────────────────────
    {
        "name"           : "listings_silver_merged",
        "silver"         : S("listings_silver_merged"),
        "quarantine"     : S("listings_main_quarantine"),
        "bronze_sources" : [B("listings_csv_copyinto"), B("listings_json_autoloader"),
                            B("listings_xml_pyspark"),  B("listings_csv_dlt")],
        "primary_key"    : ["listing_id"],
        "audit_cols"     : ["bronze_load_dt", "bronze_source_file", "silver_load_dt"],
        # listing_id is a true PK across all Bronze sources → distinct() is valid
        "t1_dedup_exprs" : [("id", "listing_id", _id_main)],
        "t1_use_simple_eq": False,
        "t2_pre_filter"  : lambda df: df.filter(
            F.col("listing_id").isNotNull() & (F.col("price_rub") > 0)
        ),
        "t2_select"      : [
            ("id",   "listing_id",   _id_main),
            ("date", "listing_date", _date),
            ("cost", "price_rub",    _price),
        ],
    },

    # ── car_catalog_transformation ────────────────────────────────────────────
    {
        "name"           : "car_catalog_transformation",
        "silver"         : S("car_catalog_transformation"),
        "quarantine"     : S("car_catalog_quarantine"),
        "bronze_sources" : [B("car_catalog")],
        "primary_key"    : ["brand", "model", "generation"],
        "audit_cols"     : ["bronze_load_dt", "bronze_source_file", "silver_load_dt"],
        
        "t1_dedup_exprs" : [
            ("Марка",             "brand",           _clean),
            ("Модель",            "model",           _clean),
            ("Поколение",         "generation",      _clean),
            ("Комплектация",      "trim_level",      _clean),
            ("Объём двигателя",   "engine_volume_l", _eng_vol),
            ("Мощность двигателя","engine_power_hp",  _eng_pow),
        ],
        "t1_use_simple_eq": True,
        "t2_pre_filter"  : lambda df: df.filter(
            F.col("brand").isNotNull() & F.col("model").isNotNull()
        ),
        "t2_select"      : [
            ("Марка",    "brand",      _clean),
            ("Модель",   "model",      _clean),
            ("Поколение","generation", _clean),
        ],
    },

    # ── listings_text_transformation ──────────────────────────────────────────
    {
        "name"           : "listings_text_transformation",
        "silver"         : S("listings_text_transformation"),
        "quarantine"     : S("listings_text_quarantine"),
        "bronze_sources" : [B("listings_text")],
        "primary_key"    : ["listing_id"],
        "audit_cols"     : ["bronze_load_dt", "bronze_source_file", "silver_load_dt"],
        "t1_dedup_exprs" : [("id", "listing_id", _id_dbl)],
        "t1_use_simple_eq": False,
        "t2_pre_filter"  : lambda df: df.filter(F.col("listing_id").isNotNull()),
        "t2_select"      : [
            ("id",   "listing_id", _id_dbl),
            ("text", "text",       _pass),
        ],
    },

    # ── listings_photo_transformation ─────────────────────────────────────────
    {
        "name"           : "listings_photo_transformation",
        "silver"         : S("listings_photo_transformation"),
        "quarantine"     : S("listings_photo_quarantine"),
        "bronze_sources" : [B("listings_photo")],
        "primary_key"    : ["listing_id"],
        "audit_cols"     : ["bronze_load_dt", "bronze_source_file", "silver_load_dt"],
        
        "t1_dedup_exprs" : [
            ("id",        "listing_id",      _id_dbl),
            ("photo_url", "photo_url_clean", _std),
        ],
        "t1_use_simple_eq": False,
        "t2_pre_filter"  : lambda df: df.filter(F.col("listing_id").isNotNull()),
        "t2_select"      : [
            ("id",        "listing_id", _id_dbl),
            ("photo_url", "photo_url",  _pass),
        ],
    },

    # ── geography_transformation ──────────────────────────────────────────────
    {
        "name"           : "geography_transformation",
        "silver"         : S("geography_transformation"),
        "quarantine"     : S("geography_quarantine"),
        "bronze_sources" : [B("geo_locations")],
        "primary_key"    : ["city_name"],
        "audit_cols"     : ["bronze_load_dt", "bronze_source_file", "silver_load_dt"],
        
        "t1_dedup_exprs" : [
            ("name_padesh",  "city_name",          _geo),
            ("greate_padesh","city_prepositional",  _pass),
            ("lat",          "latitude",            _dbl),
            ("lon",          "longitude",           _dbl),
        ],
        "t1_valid_filter": lambda df: df.filter(
            F.col("latitude").isNotNull()  &
            F.col("longitude").isNotNull() &
            F.col("latitude").between(41, 82) &
            F.col("longitude").between(19, 180)
        ),
        "t1_use_simple_eq": False,
        "t2_pre_filter"  : lambda df: df.filter(F.col("latitude").isNotNull()),
        "t2_select"      : [
            ("name_padesh","city_name",  _geo),
            ("lat",        "latitude",   _dbl),
            ("lon",        "longitude",  _dbl),
        ],
    },
]

ALL_RESULTS = []

def record(suite, name, status, detail=""):
    """Append result and print a single test line."""
    ALL_RESULTS.append({"suite": suite, "test": name, "status": status, "detail": detail})
    tag = "✓ PASSED" if status == "PASSED" else "✗ FAILED"
    print(f"  {tag} | {name:<40} | {detail}")

def _union_bronze(sources):
    """Union all Bronze source tables, tolerating missing columns."""
    df = None
    for src in sources:
        b = spark.table(src)
        df = b if df is None else df.unionByName(b, allowMissingColumns=True)
    return df

print(f"Registry loaded — {len(REGISTRY)} tables ✓")


In [0]:
# ── 4. T1 — RECONCILIATION ────────────────────────────────────────────────────
# Verifies that every Bronze row ends up in Silver or Quarantine (none lost).

def test_reconciliation(entry):
    suite = "T1 Reconciliation"
    try:
        bronze_total = sum(spark.table(s).count() for s in entry["bronze_sources"])
        silver_cnt   = spark.table(entry["silver"]).count()
        quar_cnt     = spark.table(entry["quarantine"]).count()
        actual       = silver_cnt + quar_cnt

        if entry.get("t1_use_simple_eq", False):
            dups   = bronze_total - actual
            is_ok  = dups >= 0
            detail = (f"Raw: {bronze_total:,} | Actual(S+Q): {actual:,} | "
                      f"Dups Dropped: {dups:,} | "
                      f"[Note: distinct() not used — streaming micro-batch dedup "
                      f"not equivalent to global distinct() for non-PK dedup keys]")
        else:
            df_bronze   = _union_bronze(entry["bronze_sources"])
            dedup_exprs = entry.get("t1_dedup_exprs", [])
            df_dedup    = df_bronze.select(
                [tfn(col).alias(alias) for col, alias, tfn in dedup_exprs]
            )
            unique_exp = df_dedup.distinct().count()
            dups       = bronze_total - unique_exp
            is_ok      = (actual == unique_exp)
            detail     = (f"Raw: {bronze_total:,} | Unique_Exp: {unique_exp:,} | "
                          f"Actual(S+Q): {actual:,} | Dups Dropped: {dups:,}")

        record(suite, entry["name"], "PASSED" if is_ok else "FAILED", detail)

    except Exception as ex:
        record(suite, entry["name"], "ERROR", str(ex))


print("T1 -- RECONCILIATION")
print("-" * 100)
for entry in REGISTRY:
    test_reconciliation(entry)
print("-" * 100)


In [0]:
# ── 5. T2 — ROW-TO-ROW INTEGRITY ──────────────────────────────────────────────
# Verifies that every Silver row is byte-for-byte traceable to a Bronze row.

def test_row_integrity(entry):
    suite = "T2 Row Integrity"
    try:
        df_silver     = spark.table(entry["silver"])
        df_bronze_raw = _union_bronze(entry["bronze_sources"])

        bronze_tx = df_bronze_raw.select(
            [tfn(b_col).alias(s_col) for b_col, s_col, tfn in entry["t2_select"]]
        )
        if entry.get("t2_pre_filter"):
            bronze_tx = entry["t2_pre_filter"](bronze_tx)

        bronze_scoped = bronze_tx.join(
            df_silver.select(*entry["primary_key"]),
            on=entry["primary_key"], how="left_semi"
        )

        col_list  = [s for _, s, _ in entry["t2_select"]]
        bronze_fp = (bronze_scoped
            .select([_norm(F.col(c)).alias(c) for c in col_list])
            .withColumn("fp", F.sha2(F.concat_ws("||", *col_list), 256))
            .select("fp"))
        silver_fp = (df_silver
            .select([_norm(F.col(c)).alias(c) for c in col_list])
            .withColumn("fp", F.sha2(F.concat_ws("||", *col_list), 256))
            .select("fp"))

        unmatched = silver_fp.subtract(bronze_fp).count()
        record(suite, entry["name"],
               "PASSED" if unmatched == 0 else "FAILED",
               f"Unmatched: {unmatched:,}")

    except Exception as ex:
        record(suite, entry["name"], "ERROR", str(ex))


print("T2 -- ROW-TO-ROW INTEGRITY")
print("-" * 100)
for entry in REGISTRY:
    test_row_integrity(entry)
print("-" * 100)


In [0]:
# ── 6. T3 — AUDIT COLUMNS ─────────────────────────────────────────────────────
# Verifies that all audit metadata columns declared in the registry
# are present in the Silver table schema.

def test_audit_columns(entry):
    suite = "T3 Audit Columns"
    try:
        df      = spark.table(entry["silver"])
        missing = [c for c in entry["audit_cols"] if c not in df.columns]
        record(suite, entry["name"],
               "PASSED" if not missing else "FAILED",
               f"Missing: {missing}")
    except Exception as ex:
        record(suite, entry["name"], "ERROR", str(ex))


print("T3 -- AUDIT COLUMNS")
print("-" * 100)
for entry in REGISTRY:
    test_audit_columns(entry)
print("-" * 100)


In [0]:
# ── 7. FINAL SUMMARY ──────────────────────────────────────────────────────────
failures = [r for r in ALL_RESULTS if r["status"] not in ("PASSED", "SKIP")]
total    = len(ALL_RESULTS)
passed   = sum(1 for r in ALL_RESULTS if r["status"] == "PASSED")

print("\n" + "=" * 80)
print("  SILVER LAYER TEST SUITE — FINAL SUMMARY")
print("=" * 80)

for suite, label in [
    ("T1 Reconciliation", "T1 Reconciliation"),
    ("T2 Row Integrity",  "T2 Row Integrity"),
    ("T3 Audit Columns",  "T3 Audit Columns"),
]:
    sr  = [r for r in ALL_RESULTS if r["suite"] == suite]
    p   = sum(1 for r in sr if r["status"] == "PASSED")
    t   = len(sr)
    bar = chr(9608) * p + chr(9617) * (t - p)
    print(f"  {label:<25} {p:>4}/{t:<4}  {bar}")

print("-" * 80)
if failures:
    print(f"  FAILURES ({len(failures)})")
    for r in failures:
        print(f"  [{r['suite']}] {r['test']}")
        print(f"    {r['detail']}")
else:
    print("  All tests passed.")

print("=" * 80)
overall = "ALL TESTS PASSED" if not failures else f"{len(failures)} TEST(S) FAILED"
print(f"  OVERALL : {overall}")
print(f"  Checks  : {passed}/{total} passed")
print("=" * 80)
